In [1]:
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
import numpy as np

In [ ]:
envs = ["atari_battle_zone", "atari_double_dunk", "atari_phoenix", "atari_this_game", "atari_battle_zone", "box2d_lunar_lander", "box2d_continuous_lunar_lander", "box2d_bipedal_walker", "cc_acrobot", "cc_cartpole", "cc_mountain_car", "cc_continuous_mountain_car", "cc_pendulum", "minigrid_door_key", "minigrid_empty_random", "minigrid_four_rooms", "minigrid_unlock", "brax_ant", "brax_halfcheetah", "brax_hopper", "brax_humanoid"]
algos = ["ppo", "dqn", "sac"]

data = []
for env in envs:
    for algo in algos:
        try:
            partial_data = pd.read_csv(f"arlbench_data/256_10/{env}_{algo}.csv")
            partial_data["env_name"] = env
            partial_data["algorithm"] = algo
            data.append(partial_data)
        except FileNotFoundError:
            continue
data = pd.concat(data)

ValueError: No objects to concatenate

In [ ]:
## add domain
data["domain"] = data["env_name"].apply(lambda x: x.split("_")[0])

In [ ]:
# Get unique domains
domains = data["domain"].unique()
print(f"Available domains: {domains}")

In [ ]:
# normalization cross-algorithm per environment
data["normed_performance"] = data.groupby("env_name")["performance"].transform(
    lambda x: (x - x.min()) / (x.max() - x.min() + 1e-8)
)

# on data_two we normalize per env and algorithm
data["normed_performance_per_alg"] = data.groupby(["env_name", "algorithm"])["performance"].transform(
    lambda x: (x - x.min()) / (x.max() - x.min() + 1e-8)
)

# on data_two we normalize per env and algorithm
data["_seed_avg"] = data.groupby(["env_name", "algorithm", "config_id"])["performance"].transform("mean")
data["normed_performance_per_alg_seedavg"] = data.groupby(["env_name", "algorithm"], group_keys=False).apply(
    lambda g: (g["performance"] - g["_seed_avg"].min()) / (g["_seed_avg"].max() - g["_seed_avg"].min() + 1e-8)
)
data.drop(columns=["_seed_avg"], inplace=True)

In [ ]:
def compute_rankings_counting(mu: np.ndarray, sigma: np.ndarray) -> np.ndarray:
    """
    Compute rankings based on counting overlaps, without sorting or binary search.
    
    For each configuration j, we count:
    - How many configurations have upper bound (μ + σ) below j's lower bound (μ - σ)
      → these are strictly worse (gives lower bound on rank)
    - How many configurations have lower bound (μ - σ) below j's upper bound (μ + σ)
      → these overlap or are worse (gives upper bound on rank)
    
    The rank is the average of these bounds.
    
    Args:
        mu: Array of shape (n,) with aggregate metrics for each configuration
        sigma: Array of shape (n,) with spread metrics for each configuration
    
    Returns:
        ranks: Array of shape (n,) with the computed rank for each configuration
    """
    n = len(mu)
    
    # Compute bounds for all configurations
    upper_bounds = mu + sigma  # Best case for each config
    lower_bounds = mu - sigma  # Worst case for each config
    
    # For each configuration j, compute rank bounds
    l = np.zeros(n)  # Lower rank bound (1-indexed rank)
    u = np.zeros(n)  # Upper rank bound (1-indexed rank)
    
    for j in range(n):
        # Count how many configurations are strictly better than j
        # (their lower bound is above j's upper bound)
        strictly_better = np.sum(lower_bounds > upper_bounds[j])
        
        # Count how many configurations could potentially be better or equal
        # (their upper bound is at or above j's lower bound)
        potentially_better_or_equal = np.sum(upper_bounds >= lower_bounds[j])
        
        # l[j] = number of configs strictly better + 1 (best possible rank for j)
        l[j] = strictly_better + 1
        
        # u[j] = number of configs that could be better or equal (worst possible rank for j)
        u[j] = potentially_better_or_equal
    
    # Average rank
    ranks = (l + u) / 2
    
    return ranks

In [ ]:
perf_column = "normed_performance_per_alg"

In [ ]:
# Compute rank_dict per domain
rank_dict_per_domain = {}
envs = data["env_name"].unique()
algos = data["algorithm"].unique()

for domain in domains:
    rank_dict_per_domain[domain] = {}
    domain_envs = data[data["domain"] == domain]["env_name"].unique()
    
    for algo in algos:
        rank_dict_per_domain[domain][algo] = {}
        
        for env in domain_envs:
            subset = data[(data["env_name"] == env) & (data["algorithm"] == algo)]
            
            if len(subset) == 0:
                continue

            # Identify columns starting with 'hp_config.' that have no NaN entries
            hp_config_cols = [col for col in subset.columns 
                            if col.startswith('hp_config.') and subset[col].notna().all()]

            H = [col.replace("hp_config.", "") for col in hp_config_cols]

            for i, H_i in enumerate(H):
                H_i_results = {}

                for j, h_j in enumerate(subset[f"hp_config.{H_i}"].unique().tolist()):
                    subsub = subset[subset[f"hp_config.{H_i}"] == h_j]
                    
                    H_i_results.update({h_j: {
                        "mu": subsub[perf_column].mean(),
                        "sigma": subsub[perf_column].std()
                    }})
                
                vals, mu, sigma = [], [], []
                for v, stats in H_i_results.items():
                    vals.append(v)
                    mu.append(stats["mu"])
                    sigma.append(stats["sigma"])

                vals = np.array(vals)
                mu = np.array(mu)
                sigma = np.array(sigma)

                ranks_i = compute_rankings_counting(mu, sigma)

                if H_i not in rank_dict_per_domain[domain][algo]:
                    rank_dict_per_domain[domain][algo][H_i] = {env: ranks_i}
                else:
                    rank_dict_per_domain[domain][algo][H_i].update({env: ranks_i})

In [ ]:
# Compute THCs per domain
THCs_per_domain = {}

for domain in domains:
    THCs_per_domain[domain] = {}
    
    for algo in algos:
        if algo not in rank_dict_per_domain[domain]:
            continue
            
        for H_i, env_ranks in rank_dict_per_domain[domain][algo].items():
            filtered = {env: ranks for env, ranks in env_ranks.items() 
                        if ranks.shape[0] > 0}
            
            if not filtered:
                continue
            
            # Find minimum length across all arrays
            min_len = min(ranks.shape[0] for ranks in filtered.values())
            
            # Truncate all arrays to the same length
            ranks_matrix = np.array([ranks[:min_len] for ranks in filtered.values()])
            
            ptps = np.max(ranks_matrix, axis=0) - np.min(ranks_matrix, axis=0)
            
            THCs_per_domain[domain][(algo, H_i)] = ptps / (len(ptps))

In [ ]:
# Define a color palette for domains
domain_colors = {
    'atari': '#e41a1c',
    'box2d': '#377eb8',
    'cc': '#4daf4a',
    'minigrid': '#984ea3',
    'brax': '#ff7f00'
}

In [ ]:
# Plot THC per hyperparameter for each algorithm, with bars colored by domain
for algo in algos:
    # Collect all hyperparameters for this algorithm across all domains
    all_hps = set()
    for domain in domains:
        for (a, H_i), thc in THCs_per_domain.get(domain, {}).items():
            if a == algo:
                all_hps.add(H_i)
    
    if not all_hps:
        continue
    
    # Compute average THC per HP per domain for this algorithm
    hp_domain_thcs = {}
    for H_i in all_hps:
        hp_domain_thcs[H_i] = {}
        for domain in domains:
            key = (algo, H_i)
            if key in THCs_per_domain.get(domain, {}):
                hp_domain_thcs[H_i][domain] = np.mean(THCs_per_domain[domain][key])
    
    # Sort hyperparameters by total average THC
    hp_avg = {H_i: np.mean(list(d.values())) for H_i, d in hp_domain_thcs.items() if d}
    sorted_hps = sorted(hp_avg.keys(), key=lambda x: hp_avg[x], reverse=True)
    
    # Create grouped bar plot
    fig, ax = plt.subplots(figsize=(12, len(sorted_hps) * 0.5 + 2))
    
    y_pos = np.arange(len(sorted_hps))
    bar_height = 0.15
    
    # Plot bars for each domain
    active_domains = [d for d in domains if any(d in hp_domain_thcs.get(hp, {}) for hp in sorted_hps)]
    
    for i, domain in enumerate(active_domains):
        offsets = y_pos + i * bar_height - (len(active_domains) - 1) * bar_height / 2
        values = [hp_domain_thcs.get(hp, {}).get(domain, 0) for hp in sorted_hps]
        ax.barh(offsets, values, height=bar_height, label=domain.upper(), 
                color=domain_colors.get(domain, 'gray'))
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels(sorted_hps)
    ax.invert_yaxis()
    ax.set_xlabel('THC')
    ax.set_title(f'THC per Hyperparameter by Domain - {algo.upper()}')
    ax.legend(loc='lower right')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Compute average THC per algorithm per domain
avg_thcs_per_domain = {}

for algo in algos:
    avg_thcs_per_domain[algo] = {}
    for domain in domains:
        algo_thcs = [np.mean(thc) for (a, H_i), thc in THCs_per_domain.get(domain, {}).items() if a == algo]
        if algo_thcs:
            avg_thcs_per_domain[algo][domain] = np.mean(algo_thcs)

In [ ]:
# Create grouped bar plot for average THC per algorithm, colored by domain
fig, ax = plt.subplots(figsize=(10, 6))

algo_names = [a for a in algos if a in avg_thcs_per_domain and avg_thcs_per_domain[a]]
x_pos = np.arange(len(algo_names))
bar_width = 0.15

# Get active domains (domains with data for at least one algorithm)
active_domains = [d for d in domains if any(d in avg_thcs_per_domain.get(a, {}) for a in algo_names)]

for i, domain in enumerate(active_domains):
    offsets = x_pos + i * bar_width - (len(active_domains) - 1) * bar_width / 2
    values = [avg_thcs_per_domain.get(a, {}).get(domain, 0) for a in algo_names]
    ax.bar(offsets, values, width=bar_width, label=domain.upper(), 
           color=domain_colors.get(domain, 'gray'))

ax.set_xticks(x_pos)
ax.set_xticklabels([a.upper() for a in algo_names])
ax.set_ylabel('Average THC')
ax.set_xlabel('Algorithm')
ax.set_title('Average THC per Algorithm by Domain')
ax.legend(loc='upper right')

plt.tight_layout()
plt.show()

#### Remark: Because we work with ranks, this is invariant under normalization